# 02 — MSc foundation and original results

This notebook documents the original MSc analytical foundation and connects it to the current research extension. The historical workflow is reconstructed from the dissertation appendix; it is not presented as the final modern evaluation protocol.


## 1. Original MSc dataset and problem

The MSc project used environmental IoT telemetry to predict human movement near IoT devices. The dataset contains 405,184 observations, 9 columns and 3 devices. The `motion` target contains 482 positive events and 404,702 negative events (~0.119% positive prevalence).


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
sys.path.append(str(Path.cwd().parent / 'src'))
from data import load_dataset, FEATURES, TARGET, prepare_time

df = load_dataset(Path('../data/raw/iotdata.csv'))
df = prepare_time(df)
print('Shape:', df.shape)
print('Devices:', df['device'].nunique())
print('Motion counts:')
print(df[TARGET].value_counts())
print('Positive prevalence:', df[TARGET].mean())

In [ ]:
# Original MSc-style telemetry plots
fig, axes = plt.subplots(3, 2, figsize=(12, 10))
for ax, feature in zip(axes.ravel(), FEATURES):
    ax.plot(df['datetime_utc'], df[feature], linewidth=0.5)
    ax.set_title(feature)
    ax.set_xlabel('Time')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## 2. Original preprocessing reconstruction

The dissertation code converted the Unix timestamp into hour/minute/second/microsecond fields, factorised the device identifier, and label-encoded `light` and `motion`. For historical modelling, the dissertation then sampled 482 negative observations to match the 482 positive observations before a 75/25 train/test split. This produced 722 training rows and 242 test rows.


In [ ]:
# Reconstruct the historical class-balancing counts without overwriting the canonical dataset
positive = df[df[TARGET] == True]
negative = df[df[TARGET] == False]
print('Positive rows:', len(positive))
print('Negative rows:', len(negative))
print('Historical balanced modelling rows:', len(positive) * 2)
print('Historical train rows:', 722)
print('Historical test rows:', 242)

## 3. Verified original MSc model outputs

The table below records only values recovered from the dissertation results. Accuracy is retained because it was the principal historical metric, but it should not be used as the headline metric for the current rare-event research extension.


In [ ]:
msc_results = pd.DataFrame({
    'Model': ['Random Forest', 'XGBoost', 'Logistic Regression', 'Decision Tree', 'Gradient Boosting', 'KNN', 'Gaussian NB', 'SVC'],
    'Original MSc test accuracy': [0.818182, 0.801653, 0.789256, 0.789256, 0.789256, 0.785124, 0.785124, 0.719008]
})
display(msc_results)

In [ ]:
# MSc model comparison figure
fig, ax = plt.subplots(figsize=(10, 5))
ordered = msc_results.sort_values('Original MSc test accuracy')
ax.barh(ordered['Model'], ordered['Original MSc test accuracy'] * 100)
ax.set_xlabel('Test accuracy (%)')
ax.set_title('Original MSc model comparison — reconstructed results')
ax.set_xlim(0, 100)
plt.tight_layout()
plt.show()

## 4. Random Forest result — MSc → extension

The original Random Forest implementation reported approximately 81.40% test accuracy before the later tuned comparison; the tuned model-comparison table reported approximately 81.82%. These are historical MSc results, not results from the later robustness protocol.


In [ ]:
rf_history = pd.DataFrame({
    'Stage': ['Original RandomForestClassifier()', 'Later tuned comparison'],
    'Test accuracy': [0.814050, 0.818182]
})
display(rf_history)

## 5. Why the research extension is needed

The original MSc demonstrated that environmental IoT telemetry can be used for human-movement classification under a balanced modelling subset. The extension asks a different question: **does predictive performance remain reliable when the original extreme class imbalance is preserved and evaluation conditions change across time and devices?**

The extension therefore uses the full dataset and prioritises precision-recall metrics, chronological evaluation, unseen-device testing, feature ablation, threshold sensitivity and uncertainty. The historical MSc workflow is preserved here for provenance rather than treated as the final protocol.